# Shared Stage 1 model evaluation (Google Colab)

Owner: Chua Rui Ning, Renee

This notebook evaluates validation and test in separate runs. Each split uses its own labels and model rankings, and writes reports only to its matching folder:

- Validation: Evaluation/Stage 1/validation
- Test: Evaluation/Stage 1/test

The shared article catalogue is built once from the declared train/validation and test article snapshots. Validation is for model selection. Test is for final reporting after model and reranker choices are frozen; do not use test results to tune anything.

In [7]:
%pip -q install numpy pandas pyarrow


In [8]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Configure split inputs and outputs

Set the paths in the next cell to the exact full-split top-200 rankings. Validation paths are only used with validation labels; test paths are only used with test labels. The notebook keeps outputs in separate split folders and does not reuse a split’s report or per-request file for the other split.

In [9]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd

PROJECT_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder')
EVALUATION_ROOT = PROJECT_ROOT / 'Evaluation'
STAGE1_OUTPUT_ROOT = EVALUATION_ROOT / 'Stage 1'

TRAIN_VAL_ARTICLES_PATH = PROJECT_ROOT / 'train_val' / 'articles.parquet'
TEST_ARTICLES_CANDIDATE = PROJECT_ROOT / 'test' / 'articles (1).parquet'
TEST_ARTICLES_PATH = (
    TEST_ARTICLES_CANDIDATE
    if TEST_ARTICLES_CANDIDATE.is_file()
    else PROJECT_ROOT / 'test' / 'articles.parquet'
)
ARTICLES_PATH = EVALUATION_ROOT / 'catalogue' / 'article_union.parquet'

LABELS_BY_SPLIT = {
    'validation': PROJECT_ROOT / 'feature_engineering' / 'validation_features.parquet',
    'test': PROJECT_ROOT / 'feature_engineering' / 'test_features.parquet',
}

EVALUATION_CODE_DIR = EVALUATION_ROOT / 'code'
required_modules = ('evaluate.py', 'build_catalogue_union.py')
missing_modules = [
    name for name in required_modules
    if not (EVALUATION_CODE_DIR / name).is_file()
]
if missing_modules:
    raise FileNotFoundError(
        f'Missing {missing_modules} under {EVALUATION_CODE_DIR}. '
        'Copy the evaluator Python files into Drive at Evaluation/code/.'
    )
sys.path.insert(0, str(EVALUATION_CODE_DIR))


def existing_file(path):
    path = Path(path)
    return path if path.is_file() else None


LSTM_RUN_DIR = (
    PROJECT_ROOT / 'Experiments' / '2.1_lstm' / '2.1.3_improved_run'
    / '20261104_175920_improved_seed42'
)
XGBOOST_RUN_DIR = (
    PROJECT_ROOT / 'Experiments' / '1.3_xgboost' / '20261105_small_seed42'
)
GEMINI_RUN_DIR = (
    PROJECT_ROOT / 'Experiments'
    / '3.1_pretrained_similarity_gemini_individual_history'
    / '20261105_213442_history_seed42'
)
RRF_ROOT = EVALUATION_ROOT / 'ensemble'

RUN_DEFINITIONS = [
    {
        'name': 'LSTM',
        'run_dir': LSTM_RUN_DIR,
        'catalogue_scores': True,
        'recorded_slate_scores': True,
    },
    {
        'name': 'XGBoost',
        'run_dir': XGBOOST_RUN_DIR,
        'catalogue_scores': True,
        'recorded_slate_scores': False,
    },
    {
        'name': 'Gemini',
        'run_dir': GEMINI_RUN_DIR,
        'catalogue_scores': True,
        'recorded_slate_scores': False,
    },
    {
        'name': 'RRF_lstm75_xgboost25',
        'rankings_by_split': {
            'validation': (
                RRF_ROOT / 'validation'
                / 'rrf_lstm75_xgboost25_validation_top_200.parquet'
            ),
            'test': (
                RRF_ROOT / 'test'
                / 'rrf_lstm75_xgboost25_test_top_200.parquet'
            ),
        },
        'catalogue_scores': False,
        'recorded_slate_scores': False,
    },
]


def model_runs_for_split(split):
    runs = []
    for definition in RUN_DEFINITIONS:
        if 'rankings_by_split' in definition:
            ranking_path = definition['rankings_by_split'][split]
            run_dir = None
        else:
            run_dir = definition['run_dir']
            ranking_path = (
                run_dir / 'rankings' / split / 'submission_top_200.parquet'
            )

        run_config = None
        catalogue_scores = None
        recorded_slate_scores = None
        if run_dir is not None:
            # The existing run_config is validation-scoped. A test config is
            # attached only if the exporter supplies a separate test config.
            config_name = (
                'run_config.json'
                if split == 'validation'
                else 'run_config_test.json'
            )
            run_config = existing_file(run_dir / config_name)

            if definition['catalogue_scores']:
                catalogue_scores = existing_file(
                    run_dir / 'predictions' / split
                    / 'submission_article_scores.parquet'
                )
            if definition['recorded_slate_scores']:
                recorded_slate_scores = existing_file(
                    run_dir / 'diagnostics' / 'recorded_slates'
                    / f'{split}_scores.parquet'
                )

        runs.append({
            'name': definition['name'],
            'rankings': ranking_path,
            'run_config': run_config,
            'catalogue_scores': catalogue_scores,
            'recorded_slate_scores': recorded_slate_scores,
        })
    return runs


SPLITS = ('validation', 'test')
EXPORT_K = 200
KS = [100, 200]
ALLOW_SUBSET = False
NAIVE_TIMEZONE = 'UTC'

print('Project root exists:', PROJECT_ROOT.exists())
print('Catalogue inputs:', TRAIN_VAL_ARTICLES_PATH.is_file(), TEST_ARTICLES_PATH.is_file())
print('Labels by split:')
for split, path in LABELS_BY_SPLIT.items():
    print(f'  {split}: {path} ({path.is_file()})')
print('Evaluator code:', EVALUATION_CODE_DIR)
for split in SPLITS:
    print(f'{split} ranking paths:')
    for run in model_runs_for_split(split):
        print(f"  {run['name']}: {run['rankings']} ({Path(run['rankings']).is_file()})")

Project root exists: True
Catalogue inputs: True True
Labels by split:
  validation: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/feature_engineering/validation_features.parquet (True)
  test: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/feature_engineering/test_features.parquet (True)
Evaluator code: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/code
validation ranking paths:
  LSTM: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/2.1_lstm/2.1.3_improved_run/20261104_175920_improved_seed42/rankings/validation/submission_top_200.parquet (True)
  XGBoost: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/1.3_xgboost/20261105_small_seed42/rankings/validation/submission_top_200.parquet (True)
  Gemini: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/3.1_pretrained_similarity_gemini_individual_history/20261105_213442_history_seed42

## Import shared evaluator implementation

The evaluator and catalogue builder are imported from Evaluation/code on Drive. The same evaluator code is applied independently to each split.

In [10]:
# Remove cached imports so this session uses the configured Drive modules.
for module_name in ('evaluate', 'similarity', 'build_catalogue_union'):
    sys.modules.pop(module_name, None)

from evaluate import main as evaluate_main
from build_catalogue_union import main as build_catalogue_union_main

## Build the shared article catalogue

The ordered article catalogue is shared across the evaluation runs. Rebuild it if either declared article snapshot changes. It uses train/validation precedence, then includes test-snapshot articles according to the project’s catalogue policy.

In [11]:
old_argv = sys.argv
try:
    sys.argv = [
        'build_catalogue_union.py',
        '--train-val', str(TRAIN_VAL_ARTICLES_PATH),
        '--test', str(TEST_ARTICLES_PATH),
        '--naive-timezone', NAIVE_TIMEZONE,
        '--output', str(ARTICLES_PATH),
    ]
    build_catalogue_union_main()
finally:
    sys.argv = old_argv

CATALOGUE_MANIFEST_PATH = ARTICLES_PATH.with_suffix(
    ARTICLES_PATH.suffix + '.manifest.json'
)
print('Catalogue:', ARTICLES_PATH)
print('Manifest:', CATALOGUE_MANIFEST_PATH)

{'train_val_rows': 20738, 'test_rows': 20738, 'union_rows': 20738, 'duplicate_ids_removed': 20738, 'overlap_ids': 20738, 'catalogue_id_fingerprint_sha256': 'eff069a2586ad6cd00273b1132999f38c98b1339424040d2af345c01121292c6', 'catalogue_content_fingerprint_sha256': 'bbd9dfd867eebdaef43598653d03af2ad3a8f4dc5a7d4ac11b210eb97d7e0728', 'text_flag_sources': [], 'image_flag_sources': [], 'output': '/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/catalogue/article_union.parquet', 'manifest': '/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/catalogue/article_union.parquet.manifest.json'}
Catalogue: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/catalogue/article_union.parquet
Manifest: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/catalogue/article_union.parquet.manifest.json


## Evaluate validation and test separately

Each split is evaluated only with its own labels and ranking exports. Reports and per-request files are written under that split’s Stage 1 folder. Test run configs are omitted unless a test-specific config exists; the evaluator still checks article eligibility and exact request coverage.

In [12]:
required_paths = [
    ARTICLES_PATH,
    TRAIN_VAL_ARTICLES_PATH,
    TEST_ARTICLES_PATH,
    *LABELS_BY_SPLIT.values(),
]
missing = [str(path) for path in required_paths if not Path(path).is_file()]
if missing:
    raise FileNotFoundError(
        'Required file(s) not found:\n' + '\n'.join(missing)
    )


def evaluator_compatible_run_config(run, split, output_dir):
    """Normalize supported catalogue-fingerprint aliases for one split."""
    source = run.get('run_config')
    if source is None:
        return None

    source = Path(source)
    if not source.is_file():
        raise FileNotFoundError(
            f"{run['name']} {split} run_config not found: {source}"
        )
    config = json.loads(source.read_text(encoding='utf-8'))
    eligibility = config.get('eligibility', {})
    needs_normalization = False

    if not config.get('catalogue_id_fingerprint_sha256'):
        shared_hash = eligibility.get('shared_catalogue_id_fingerprint_sha256')
        short_hash = eligibility.get('catalogue_fingerprint')
        if shared_hash:
            config['catalogue_id_fingerprint_sha256'] = shared_hash
            needs_normalization = True
        elif short_hash:
            if not CATALOGUE_MANIFEST_PATH.is_file():
                raise FileNotFoundError(
                    'Catalogue manifest is required to verify the shortened '
                    f'fingerprint for {run["name"]}.'
                )
            manifest = json.loads(
                CATALOGUE_MANIFEST_PATH.read_text(encoding='utf-8')
            )
            current_hash = manifest.get('catalogue_id_fingerprint_sha256')
            current_count = manifest.get('union_rows')
            if not current_hash or not current_hash.startswith(str(short_hash)):
                raise ValueError(
                    f"{run['name']} shortened catalogue fingerprint does not "
                    'match the current catalogue.'
                )
            declared_count = eligibility.get('catalogue_articles')
            if (
                declared_count is not None
                and current_count is not None
                and int(declared_count) != int(current_count)
            ):
                raise ValueError(
                    f"{run['name']} run_config declares {declared_count} "
                    f'catalogue articles; current catalogue has {current_count}.'
                )
            config['catalogue_id_fingerprint_sha256'] = current_hash
            needs_normalization = True
        else:
            warnings.warn(
                f"{run['name']} {split}: run_config has no catalogue ID "
                'fingerprint; skipping run_config verification for this model.'
            )
            return None

    declared_split = config.get('split')
    scope_split = config.get('scope', {}).get('split')
    declared_split = declared_split or scope_split
    if declared_split and declared_split != split:
        raise ValueError(
            f"{run['name']} run_config is for {declared_split}, not {split}. "
            'Supply the correct split-specific config or omit run_config.'
        )

    if needs_normalization:
        policy = eligibility.get('policy')
        if policy:
            config['pool_policy'] = {'name': policy}
        config['split'] = split
        exports = config.get('exports', {}).get(split, {})
        scope = config.get('scope', {})
        export_count = (
            exports.get('requests')
            or scope.get('export_requests')
        )
        if export_count is not None:
            config['export_request_scope'] = {
                'request_count': int(export_count)
            }
        normalized_path = (
            output_dir / f"{run['name'].lower().replace(' ', '_')}"
            '_evaluator_run_config.json'
        )
        normalized_path.write_text(
            json.dumps(config, indent=2) + '\n',
            encoding='utf-8',
        )
        return normalized_path
    return source


reports_by_split = {}
for split in SPLITS:
    labels_path = LABELS_BY_SPLIT[split]
    output_dir = STAGE1_OUTPUT_ROOT / split
    output_dir.mkdir(parents=True, exist_ok=True)
    model_runs = model_runs_for_split(split)

    split_required = [labels_path]
    split_required.extend(Path(run['rankings']) for run in model_runs)
    missing = [str(path) for path in split_required if not path.is_file()]
    if missing:
        raise FileNotFoundError(
            f'{split} input(s) missing:\n' + '\n'.join(missing)
        )

    reports = []
    print(f'\n===== Evaluating {split.upper()} only =====')
    for run in model_runs:
        ranking_path = Path(run['rankings'])
        stem = run['name'].lower().replace(' ', '_')
        report_path = output_dir / f'{stem}_report.json'
        per_request_path = output_dir / f'{stem}_per_request.parquet'

        args = [
            'evaluate.py',
            '--split', split,
            '--labels', str(labels_path),
            '--articles', str(ARTICLES_PATH),
            '--rankings', str(ranking_path),
            '--export-k', str(EXPORT_K),
            '--ks', *[str(k) for k in KS],
            '--output', str(report_path),
            '--per-request-output', str(per_request_path),
        ]

        run_config = evaluator_compatible_run_config(
            run, split, output_dir
        )
        optional_inputs = [
            ('catalogue_scores', '--catalogue-scores'),
            ('recorded_slate_scores', '--recorded-slate-scores'),
        ]
        if run_config is not None:
            args.extend(['--run-config', str(run_config)])
        for key, flag in optional_inputs:
            value = run.get(key)
            if value is not None:
                value = Path(value)
                if not value.is_file():
                    raise FileNotFoundError(
                        f"{run['name']} {split} {key} not found: {value}"
                    )
                args.extend([flag, str(value)])

        if ALLOW_SUBSET:
            args.append('--allow-subset')

        old_argv = sys.argv
        try:
            sys.argv = args
            print(f"Evaluating {run['name']} on {split}...")
            evaluate_main()
        finally:
            sys.argv = old_argv

        report = json.loads(report_path.read_text(encoding='utf-8'))
        if report.get('split') != split:
            raise RuntimeError(
                f'{run["name"]}: evaluator reported split '
                f'{report.get("split")!r}, expected {split!r}'
            )
        reports.append({'model': run['name'], **report['metrics']})
        print(
            'Pool check:',
            'run-config verified'
            if 'run_config' in report.get('inputs', {})
            else 'not verified by run_config; ranking eligibility checked',
        )

    reports_by_split[split] = pd.DataFrame(reports)
    display(reports_by_split[split])
    print(f'{split} reports saved under: {output_dir}')


===== Evaluating VALIDATION only =====
Evaluating LSTM on validation...
Evaluated 1,000/29,110 requests
Evaluated 2,000/29,110 requests
Evaluated 3,000/29,110 requests
Evaluated 4,000/29,110 requests
Evaluated 5,000/29,110 requests
Evaluated 6,000/29,110 requests
Evaluated 7,000/29,110 requests
Evaluated 8,000/29,110 requests
Evaluated 9,000/29,110 requests
Evaluated 10,000/29,110 requests
Evaluated 11,000/29,110 requests
Evaluated 12,000/29,110 requests
Evaluated 13,000/29,110 requests
Evaluated 14,000/29,110 requests
Evaluated 15,000/29,110 requests
Evaluated 16,000/29,110 requests
Evaluated 17,000/29,110 requests
Evaluated 18,000/29,110 requests
Evaluated 19,000/29,110 requests
Evaluated 20,000/29,110 requests
Evaluated 21,000/29,110 requests
Evaluated 22,000/29,110 requests
Evaluated 23,000/29,110 requests
Evaluated 24,000/29,110 requests
Evaluated 25,000/29,110 requests
Evaluated 26,000/29,110 requests
Evaluated 27,000/29,110 requests
Evaluated 28,000/29,110 requests
Evaluated 29

/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/code/evaluate.py:1240: UserWarning: Run config has no catalogue content fingerprint; metadata identity is verified only by article IDs
  warnings.warn("Run config has no catalogue content fingerprint; metadata identity is verified only by article IDs")
/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/code/evaluate.py:1245: UserWarning: Run config has no split field; split identity is taken from --split and the labels
  warnings.warn("Run config has no split field; split identity is taken from --split and the labels")


{
  "protocol": "schema_v4_catalogue_and_stage2_evaluation",
  "split": "validation",
  "scope": "validation_development",
  "export_k": 200,
  "ks": [
    100,
    200
  ],
  "primary_metric": "known_click_recall@100",
  "selection_tiebreaker": "observed_click_ndcg@100",
  "metrics": {
    "requests": 29110,
    "label_requests_available": 29110,
    "requests_without_export": 0,
    "requests_with_known_clicks": 29110,
    "requests_without_known_clicks": 0,
    "users": 1719,
    "logged_candidate_rows": 344666,
    "ranked_catalogue_rows": 5822000,
    "shortlist_length_min_median_max": [
      200,
      200.0,
      200
    ],
    "empty_pool_requests": 0,
    "known_clicks": 29249,
    "eligible_known_clicks": 29249,
    "eligibility_coverage_of_known_clicks": 1.0,
    "click_exclusions": {
      "not_in_catalogue": 0,
      "missing_publication_time": 0,
      "published_after_request": 0
    },
    "known_click_recall@100": 0.9161032863849765,
    "known_click_recall@100_eligi

/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/code/evaluate.py:1240: UserWarning: Run config has no catalogue content fingerprint; metadata identity is verified only by article IDs
  warnings.warn("Run config has no catalogue content fingerprint; metadata identity is verified only by article IDs")
/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/code/evaluate.py:1254: UserWarning: Run config has no export_request_scope; exact scope is recorded from evaluated rankings
  warnings.warn("Run config has no export_request_scope; exact scope is recorded from evaluated rankings")


{
  "protocol": "schema_v4_catalogue_and_stage2_evaluation",
  "split": "validation",
  "scope": "validation_development",
  "export_k": 200,
  "ks": [
    100,
    200
  ],
  "primary_metric": "known_click_recall@100",
  "selection_tiebreaker": "observed_click_ndcg@100",
  "metrics": {
    "requests": 29110,
    "label_requests_available": 29110,
    "requests_without_export": 0,
    "requests_with_known_clicks": 29110,
    "requests_without_known_clicks": 0,
    "users": 1719,
    "logged_candidate_rows": 344666,
    "ranked_catalogue_rows": 5822000,
    "shortlist_length_min_median_max": [
      200,
      200.0,
      200
    ],
    "empty_pool_requests": 0,
    "known_clicks": 29249,
    "eligible_known_clicks": 29249,
    "eligibility_coverage_of_known_clicks": 1.0,
    "click_exclusions": {
      "not_in_catalogue": 0,
      "missing_publication_time": 0,
      "published_after_request": 0
    },
    "known_click_recall@100": 0.8582268407191114,
    "known_click_recall@100_eligi

/tmp/ipykernel_4171/1115982505.py:64: UserWarning: Gemini validation: run_config has no catalogue ID fingerprint; skipping run_config verification for this model.
  warnings.warn(


Evaluating Gemini on validation...
Evaluated 1,000/29,110 requests
Evaluated 2,000/29,110 requests
Evaluated 3,000/29,110 requests
Evaluated 4,000/29,110 requests
Evaluated 5,000/29,110 requests
Evaluated 6,000/29,110 requests
Evaluated 7,000/29,110 requests
Evaluated 8,000/29,110 requests
Evaluated 9,000/29,110 requests
Evaluated 10,000/29,110 requests
Evaluated 11,000/29,110 requests
Evaluated 12,000/29,110 requests
Evaluated 13,000/29,110 requests
Evaluated 14,000/29,110 requests
Evaluated 15,000/29,110 requests
Evaluated 16,000/29,110 requests
Evaluated 17,000/29,110 requests
Evaluated 18,000/29,110 requests
Evaluated 19,000/29,110 requests
Evaluated 20,000/29,110 requests
Evaluated 21,000/29,110 requests
Evaluated 22,000/29,110 requests
Evaluated 23,000/29,110 requests
Evaluated 24,000/29,110 requests
Evaluated 25,000/29,110 requests
Evaluated 26,000/29,110 requests
Evaluated 27,000/29,110 requests
Evaluated 28,000/29,110 requests
Evaluated 29,000/29,110 requests
Evaluated 29,110/

,model,requests,label_requests_available,requests_without_export,requests_with_known_clicks,requests_without_known_clicks,users,logged_candidate_rows,ranked_catalogue_rows,shortlist_length_min_median_max,...,distinct_categories@100,median_article_age_h@200,unknown_age_share@200,age_le_24h_share@200,age_24_72h_share@200,age_gt_72h_share@200,topic_coverage@200,topic_article_coverage@200,intra_list_diversity@200,distinct_categories@200
0,LSTM,29110,29110,0,29110,0,1719,344666,5822000,"[200, 200.0, 200]",...,10.323703,19.453696,0.0,0.608508,0.387659,0.003832,64.679114,0.999768,0.901410,12.092133
1,XGBoost,29110,29110,0,29110,0,1719,344666,5822000,"[200, 200.0, 200]",...,9.577087,56.967367,0.0,0.573261,0.112754,0.313985,59.919134,0.999598,0.845249,11.020508
2,Gemini,29110,29110,0,29110,0,1719,344666,5822000,"[200, 200.0, 200]",...,6.677156,2014.705364,0.0,0.015848,0.039117,0.945035,44.474064,0.999185,0.783344,7.908245
3,RRF_lstm75_xgboost25,29110,29110,0,29110,0,1719,344666,5822000,"[200, 200.0, 200]",...,10.052834,19.453401,0.0,0.608480,0.386566,0.004954,64.674476,0.999769,0.901493,12.099622


validation reports saved under: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/Stage 1/validation

===== Evaluating TEST only =====
Evaluating LSTM on test...
Evaluated 1,000/29,142 requests
Evaluated 2,000/29,142 requests
Evaluated 3,000/29,142 requests
Evaluated 4,000/29,142 requests
Evaluated 5,000/29,142 requests
Evaluated 6,000/29,142 requests
Evaluated 7,000/29,142 requests
Evaluated 8,000/29,142 requests
Evaluated 9,000/29,142 requests
Evaluated 10,000/29,142 requests
Evaluated 11,000/29,142 requests
Evaluated 12,000/29,142 requests
Evaluated 13,000/29,142 requests
Evaluated 14,000/29,142 requests
Evaluated 15,000/29,142 requests
Evaluated 16,000/29,142 requests
Evaluated 17,000/29,142 requests
Evaluated 18,000/29,142 requests
Evaluated 19,000/29,142 requests
Evaluated 20,000/29,142 requests
Evaluated 21,000/29,142 requests
Evaluated 22,000/29,142 requests
Evaluated 23,000/29,142 requests
Evaluated 24,000/29,142 requests
Evaluated 25,000/29,142 requ

,model,requests,label_requests_available,requests_without_export,requests_with_known_clicks,requests_without_known_clicks,users,logged_candidate_rows,ranked_catalogue_rows,shortlist_length_min_median_max,...,distinct_categories@100,median_article_age_h@200,unknown_age_share@200,age_le_24h_share@200,age_24_72h_share@200,age_gt_72h_share@200,topic_coverage@200,topic_article_coverage@200,intra_list_diversity@200,distinct_categories@200
0,LSTM,29142,29142,0,29142,0,1757,348951,5828400,"[200, 200.0, 200]",...,10.342084,19.509819,0.0,0.607023,0.389064,0.003913,64.701942,0.999753,0.902121,12.093851
1,XGBoost,29142,29142,0,29142,0,1757,348951,5828400,"[200, 200.0, 200]",...,9.543065,72.533866,0.0,0.570672,0.112153,0.317175,59.885698,0.999615,0.846965,11.034967
2,Gemini,29142,29142,0,29142,0,1757,348951,5828400,"[200, 200.0, 200]",...,6.733992,2008.646383,0.0,0.015565,0.039002,0.945433,44.723972,0.999038,0.784379,8.000137
3,RRF_lstm75_xgboost25,29142,29142,0,29142,0,1757,348951,5828400,"[200, 200.0, 200]",...,10.069007,19.510487,0.0,0.606926,0.386921,0.006153,64.687702,0.999755,0.902250,12.104866


test reports saved under: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/Stage 1/test


## Verify eligible-catalogue counts by split

This checks each split’s saved per-request outputs against the shared catalogue’s publication timestamps. The inputs and outputs remain split-specific; validation per-request files are never read as test files, or vice versa.

In [13]:
published_ns = (
    pd.to_datetime(
        pd.read_parquet(ARTICLES_PATH, columns=['published_time'])['published_time'],
        utc=True,
    )
    .dropna()
    .sort_values()
    .astype('datetime64[ns, UTC]')
    .astype('int64')
    .to_numpy()
)

for split in SPLITS:
    split_output_dir = STAGE1_OUTPUT_ROOT / split
    for run in model_runs_for_split(split):
        stem = run['name'].lower().replace(' ', '_')
        per_request_path = (
            split_output_dir / f'{stem}_per_request.parquet'
        )
        if not per_request_path.is_file():
            print(f'{split} {run["name"]}: per-request output missing')
            continue

        per = pd.read_parquet(
            per_request_path,
            columns=['impression_time', 'eligible_catalogue_count'],
        )
        times_ns = (
            pd.to_datetime(per['impression_time'], utc=True)
            .astype('datetime64[ns, UTC]')
            .astype('int64')
            .to_numpy()
        )
        expected_counts = np.searchsorted(
            published_ns,
            times_ns,
            side='right',
        )
        actual_counts = per['eligible_catalogue_count'].to_numpy(
            dtype='int64'
        )
        mismatched = actual_counts != expected_counts
        print({
            'split': split,
            'model': run['name'],
            'requests_checked': int(len(per)),
            'mismatched_eligible_counts': int(mismatched.sum()),
            'actual_min_max': (
                [int(actual_counts.min()), int(actual_counts.max())]
                if len(actual_counts) else None
            ),
            'expected_min_max': (
                [int(expected_counts.min()), int(expected_counts.max())]
                if len(expected_counts) else None
            ),
        })
        if mismatched.any():
            display(per.loc[mismatched].head(10))

{'split': 'validation', 'model': 'LSTM', 'requests_checked': 29110, 'mismatched_eligible_counts': 0, 'actual_min_max': [19007, 19871], 'expected_min_max': [19007, 19871]}
{'split': 'validation', 'model': 'XGBoost', 'requests_checked': 29110, 'mismatched_eligible_counts': 0, 'actual_min_max': [19007, 19871], 'expected_min_max': [19007, 19871]}
{'split': 'validation', 'model': 'Gemini', 'requests_checked': 29110, 'mismatched_eligible_counts': 0, 'actual_min_max': [19007, 19871], 'expected_min_max': [19007, 19871]}
{'split': 'validation', 'model': 'RRF_lstm75_xgboost25', 'requests_checked': 29110, 'mismatched_eligible_counts': 0, 'actual_min_max': [19007, 19871], 'expected_min_max': [19007, 19871]}
{'split': 'test', 'model': 'LSTM', 'requests_checked': 29142, 'mismatched_eligible_counts': 0, 'actual_min_max': [19007, 19871], 'expected_min_max': [19007, 19871]}
{'split': 'test', 'model': 'XGBoost', 'requests_checked': 29142, 'mismatched_eligible_counts': 0, 'actual_min_max': [19007, 19871]

In [14]:
metric_columns = [
    'known_click_recall@100',
    'observed_click_ndcg@100',
    'known_click_recall@200',
    'observed_click_ndcg@200',
    'age_le_24h_share@100',
    'age_le_24h_share@200',
    'median_article_age_h@100',
    'median_article_age_h@200',
    'shared_intra_list_diversity@100',
    'shared_intra_list_diversity@200',
    'distinct_categories@100',
    'distinct_categories@200',
]

stage1_summaries = {}
for split in SPLITS:
    report_dir = STAGE1_OUTPUT_ROOT / split
    rows = []

    for report_path in sorted(report_dir.glob('*_report.json')):
        report = json.loads(report_path.read_text(encoding='utf-8'))
        if report.get('split') != split:
            raise ValueError(
                f'{report_path} contains split {report.get("split")!r}; '
                f'expected {split!r}'
            )

        metrics = report.get('metrics', {})
        rows.append({
            'model': report_path.stem.removesuffix('_report'),
            'split': split,
            'requests': report.get('evaluated_request_count'),
            **{
                metric: metrics.get(metric)
                for metric in metric_columns
            },
        })

    if not rows:
        raise FileNotFoundError(
            f'No {split} report JSON files found in {report_dir}'
        )

    summary = pd.DataFrame(rows).sort_values(
        'known_click_recall@100',
        ascending=False,
    )
    csv_path = report_dir / 'stage1_model_metrics_summary.csv'
    summary.to_csv(csv_path, index=False)
    stage1_summaries[split] = summary

    print(f'{split.upper()} summary saved to: {csv_path}')
    display(summary.round(4))

VALIDATION summary saved to: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/Stage 1/validation/stage1_model_metrics_summary.csv


,model,split,requests,known_click_recall@100,observed_click_ndcg@100,known_click_recall@200,observed_click_ndcg@200,age_le_24h_share@100,age_le_24h_share@200,median_article_age_h@100,median_article_age_h@200,shared_intra_list_diversity@100,shared_intra_list_diversity@200,distinct_categories@100,distinct_categories@200
1,lstm,validation,29110,0.9161,0.2861,0.9477,0.2905,0.9556,0.6085,10.0640,19.4537,None,None,10.3237,12.0921
2,rrf_lstm75_xgboost25,validation,29110,0.9143,0.3079,0.9477,0.3126,0.9825,0.6085,10.0082,19.4534,None,None,10.0528,12.0996
3,xgboost,validation,29110,0.8582,0.2212,0.9227,0.2304,0.8953,0.5733,84.9927,56.9674,None,None,9.5771,11.0205
0,gemini,validation,29110,0.0230,0.0049,0.0363,0.0068,0.0216,0.0158,1322.9919,2014.7054,None,None,6.6772,7.9082


TEST summary saved to: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/Stage 1/test/stage1_model_metrics_summary.csv


,model,split,requests,known_click_recall@100,observed_click_ndcg@100,known_click_recall@200,observed_click_ndcg@200,age_le_24h_share@100,age_le_24h_share@200,median_article_age_h@100,median_article_age_h@200,shared_intra_list_diversity@100,shared_intra_list_diversity@200,distinct_categories@100,distinct_categories@200
1,lstm,test,29142,0.9167,0.2882,0.9465,0.2924,0.9541,0.6070,10.0705,19.5098,None,None,10.3421,12.0939
2,rrf_lstm75_xgboost25,test,29142,0.9150,0.3112,0.9464,0.3157,0.9813,0.6069,10.0110,19.5105,None,None,10.0690,12.1049
3,xgboost,test,29142,0.8583,0.2220,0.9222,0.2311,0.8850,0.5707,118.8371,72.5339,None,None,9.5431,11.0350
0,gemini,test,29142,0.0228,0.0048,0.0359,0.0067,0.0211,0.0156,1372.0224,2008.6464,None,None,6.7340,8.0001


In [15]:
import json
import pandas as pd

for split in ("validation", "test"):
    report_dir = STAGE1_OUTPUT_ROOT / split
    report_paths = sorted(report_dir.glob("*_report.json"))

    if not report_paths:
        raise FileNotFoundError(
            f"No evaluation reports found for {split} in {report_dir}"
        )

    rows = []
    for report_path in report_paths:
        report = json.loads(report_path.read_text(encoding="utf-8"))

        # Prevent a report from accidentally appearing in the wrong split's CSV.
        if report.get("split") != split:
            raise ValueError(
                f"{report_path} is marked {report.get('split')!r}, "
                f"but is in the {split!r} folder."
            )

        metrics = pd.json_normalize(
            report.get("metrics", {}),
            sep=".",
        ).to_dict(orient="records")[0]

        rows.append({
            "model": report_path.stem.removesuffix("_report"),
            "split": split,
            "requests": report.get("evaluated_request_count"),
            **metrics,
        })

    summary = pd.DataFrame(rows)

    # Put common relevance metrics first when they exist; retain every other metric too.
    first_columns = [
        col for col in [
            "model",
            "split",
            "requests",
            "known_click_recall@100",
            "observed_click_ndcg@100",
            "known_click_recall@200",
            "observed_click_ndcg@200",
        ]
        if col in summary.columns
    ]
    remaining_columns = [
        col for col in summary.columns if col not in first_columns
    ]
    summary = summary[first_columns + remaining_columns]

    output_path = report_dir / "all_model_metrics_summary.csv"
    summary.to_csv(output_path, index=False)

    print(f"{split.title()} metrics saved to: {output_path}")
    display(summary)

Validation metrics saved to: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/Stage 1/validation/all_model_metrics_summary.csv


,model,split,requests,known_click_recall@100,observed_click_ndcg@100,known_click_recall@200,observed_click_ndcg@200,label_requests_available,requests_without_export,requests_with_known_clicks,...,age_le_24h_share@200,age_24_72h_share@200,age_gt_72h_share@200,topic_coverage@200,topic_article_coverage@200,intra_list_diversity@200,distinct_categories@200,click_exclusions.not_in_catalogue,click_exclusions.missing_publication_time,click_exclusions.published_after_request
0,gemini,validation,29110,0.023033,0.004935,0.036345,0.006788,29110,0,29110,...,0.015848,0.039117,0.945035,44.474064,0.999185,0.783344,7.908245,0,0,0
1,lstm,validation,29110,0.916103,0.286070,0.947656,0.290513,29110,0,29110,...,0.608508,0.387659,0.003832,64.679114,0.999768,0.901410,12.092133,0,0,0
2,rrf_lstm75_xgboost25,validation,29110,0.914283,0.307936,0.947656,0.312644,29110,0,29110,...,0.608480,0.386566,0.004954,64.674476,0.999769,0.901493,12.099622,0,0,0
3,xgboost,validation,29110,0.858227,0.221182,0.922665,0.230362,29110,0,29110,...,0.573261,0.112754,0.313985,59.919134,0.999598,0.845249,11.020508,0,0,0


Test metrics saved to: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/Stage 1/test/all_model_metrics_summary.csv


,model,split,requests,known_click_recall@100,observed_click_ndcg@100,known_click_recall@200,observed_click_ndcg@200,label_requests_available,requests_without_export,requests_with_known_clicks,...,age_le_24h_share@200,age_24_72h_share@200,age_gt_72h_share@200,topic_coverage@200,topic_article_coverage@200,intra_list_diversity@200,distinct_categories@200,click_exclusions.not_in_catalogue,click_exclusions.missing_publication_time,click_exclusions.published_after_request
0,gemini,test,29142,0.022806,0.004837,0.035880,0.006662,29142,0,29142,...,0.015565,0.039002,0.945433,44.723972,0.999038,0.784379,8.000137,0,0,0
1,lstm,test,29142,0.916706,0.288209,0.946507,0.292400,29142,0,29142,...,0.607023,0.389064,0.003913,64.701942,0.999753,0.902121,12.093851,0,0,0
2,rrf_lstm75_xgboost25,test,29142,0.914956,0.311234,0.946439,0.315666,29142,0,29142,...,0.606926,0.386921,0.006153,64.687702,0.999755,0.902250,12.104866,0,0,0
3,xgboost,test,29142,0.858251,0.221980,0.922174,0.231109,29142,0,29142,...,0.570672,0.112153,0.317175,59.885698,0.999615,0.846965,11.034967,0,0,0


## Separation and usage notes

- Validation labels and rankings are read only by the validation evaluation loop and written to Evaluation/Stage 1/validation.
- Test labels and rankings are read only by the test evaluation loop and written to Evaluation/Stage 1/test.
- The shared catalogue is built once from the declared article snapshots and used by both splits under the same eligibility policy.
- Test run configs are optional and must be specifically for test. The validation run_config is never reused for test.
- Freeze model and reranker choices before interpreting the final test metrics. Do not tune using test results.
- Set ALLOW_SUBSET = True only for a declared sampled run with exact matching request IDs.